# ReID Google Colab Runner

This notebook runs the `LampOfSocrates/reid` project in Google Colab.

In Colab, the mounted Drive path that corresponds to `G:\\My Drive\\Colab Notebooks\\...` is typically `/content/drive/MyDrive/Colab Notebooks/...`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/LampOfSocrates/reid.git"
REPO_ROOT = Path("/content/reid")

if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")

In [ ]:
import sys
import subprocess

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "timm",
        "pandas",
        "matplotlib",
        "pillow",
        "tqdm"
    ],
    check=True,
)

print("Notebook dependencies are ready.")

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/Colab Notebooks")
DATA_ROOT = DRIVE_ROOT / "data" / "VeRI"
OUTPUT_ROOT = DRIVE_ROOT / "reid_runs"

EXPERIMENT_PRESETS = {
    "bot_baseline": {
        "model": "bot",
        "epochs": 24,
        "batch_size": 64,
        "learning_rate": 0.00035,
        "weight_decay": 5e-4,
        "run_tag": "baseline",
    },
    "agw_fast": {
        "model": "agw",
        "epochs": 8,
        "batch_size": 32,
        "learning_rate": 0.00035,
        "weight_decay": 5e-4,
        "run_tag": "fast-check",
    },
    "transreid_baseline": {
        "model": "transreid",
        "epochs": 24,
        "batch_size": 32,
        "learning_rate": 0.00035,
        "weight_decay": 5e-4,
        "run_tag": "baseline",
    },
    "pcb_baseline": {
        "model": "pcb",
        "epochs": 24,
        "batch_size": 64,
        "learning_rate": 0.00035,
        "weight_decay": 5e-4,
        "run_tag": "baseline",
    },
    "clip_senet_baseline": {
        "model": "clip_senet",
        "epochs": 24,
        "batch_size": 32,
        "learning_rate": 0.00035,
        "weight_decay": 5e-4,
        "run_tag": "baseline",
    },
}

SELECTED_EXPERIMENT = "bot_baseline"
USE_GPU = True
NUM_WORKERS = 2

experiment = dict(EXPERIMENT_PRESETS[SELECTED_EXPERIMENT])
RUN_TAG = experiment.pop("run_tag", SELECTED_EXPERIMENT)

train_data_dir = DATA_ROOT / "image_train"
assert train_data_dir.exists(), f"Expected training data at {train_data_dir}"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Selected experiment: {SELECTED_EXPERIMENT}")
print(f"Training data dir: {train_data_dir}")
print(f"Output root: {OUTPUT_ROOT}")
experiment

In [ ]:
from notebook_runner import run_experiment

result = run_experiment(
    model_name=experiment["model"],
    epochs=experiment["epochs"],
    batch_size=experiment["batch_size"],
    data_dir=train_data_dir,
    output_root=OUTPUT_ROOT,
    learning_rate=experiment["learning_rate"],
    run_tag=RUN_TAG,
    use_gpu=USE_GPU,
)

{key: str(value) for key, value in result.items()}

In [ ]:
from IPython.display import Image, display
import pandas as pd

metrics_df = pd.read_csv(result["metrics_csv"])
display(metrics_df.tail())
display(Image(filename=str(result["plot_path"])))

print("Run folder contents:")
for path in sorted(result["run_dir"].iterdir()):
    print(path.name)